In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import torch
import torch.nn as nn

from nichejepa.models.gene_transformer import *

from nichejepa.utils.tensors import (repeat_interleave_batch,
                                     trunc_normal_,)

In [ ]:
pos_emb = get_1d_sincos_pos_embed_from_grid(embed_dim=4,
                                            pos=np.arange(5, dtype=float))
pos_emb

In [ ]:
pos_emb = get_1d_sincos_pos_embed(embed_dim=4,
                                  grid_size=5,
                                  cls_token=True)
pos_emb

In [ ]:
drop_prob = 0.1
training = True

x = torch.tensor([[12, 5, 4, 28, 64, 32, 16, 8, 55, 3],
                  [4, 3, 12, 16, 22, 5, 44, 33, 14, 24]])

drop_path(x,
          drop_prob=drop_prob,
          training=training)

### GeneTransformerEncoder Forward Pass

In [ ]:
vocab_size = 100
embed_dim = 4
seq_len = 10
depth = 12
pos_learnable = False
has_cls = True
drop_path_rate = 0.0
init_std = 0.02

In [ ]:
# Initialize gene embeddings
gene_embed = nn.Embedding(
    vocab_size + (1 if has_cls else 0), # vocab_size incl. <pad>
    embed_dim,
    padding_idx=0)
gene_embed(torch.tensor([99]))

In [ ]:
# Initialize segment embeddings
seg_embed = nn.Embedding(2 + 1, # incl. <pad>
                         embed_dim,
                         padding_idx=0)
seg_embed(torch.tensor([0, 1, 2]))

In [ ]:
# Retrieve positional embeddings
if pos_learnable:
    pos_embed = nn.Parameter(
        torch.zeros(1, seq_len + (1 if has_cls else 0), embed_dim),
        requires_grad=True)
    trunc_normal_(pos_embed, std=init_std)
    if has_cls:
        pos_embed.data[0, 0, :] = 0
else:
    pos_embed = nn.Parameter(
        torch.zeros(1, seq_len + (1 if has_cls else 0), embed_dim),
        requires_grad=False)
    pos_embed_temp = get_1d_sincos_pos_embed(pos_embed.shape[-1],
                                             seq_len,
                                             cls_token=has_cls)
    pos_embed.data.copy_(
        torch.from_numpy(pos_embed_temp).float().unsqueeze(0))
pos_embed

In [ ]:
dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
dpr

### GeneTransformerEncoder Embedding Retrieval

In [ ]:
vocab_size = 100
embed_dim = 12
seq_len = 10
depth = 12
pos_learnable = True
has_cls = True

In [ ]:
encoder = GeneTransformerEncoder(vocab_size=vocab_size,
                                 embed_dim=embed_dim,
                                 seq_len=seq_len,
                                 pos_learnable=pos_learnable,
                                 has_cls=has_cls)

In [ ]:
if has_cls:
    x = torch.tensor([[100, 12, 5, 4, 28, 64, 32, 16, 8, 55, 3],
                      [100, 4, 3, 12, 16, 22, 5, 44, 33, 14, 24]])
    seg_label = torch.tensor([[0, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2],
                              [0, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2]])
else:
    x = torch.tensor([[12, 5, 4, 28, 64, 32, 16, 8, 55, 3],
                      [4, 3, 12, 16, 22, 5, 44, 33, 14, 24]])
    seg_label = torch.tensor([[1, 1, 1, 1, 1, 2, 2, 2, 2, 2],
                              [1, 1, 1, 1, 1, 2, 2, 2, 2, 2]])

In [ ]:
pos_embeds = encoder.return_pos_emb(x)
print(pos_embeds[0].shape)
print(pos_embeds[0])

In [ ]:
gene_embeds = encoder.return_gene_emb(x)
print(gene_embeds[0].shape)
print(gene_embeds[0])

In [ ]:
embeds = encoder.return_multi_layer_emb(x,
                                        seg_label)
print(len(embeds))
print(embeds[-1].shape)
print(embeds[-1])